In [0]:
# ============================================
# RAILWAY DATA ENGINEERING PROJECT
# GOLD LAYER
# ============================================

from pyspark.sql.functions import *

# Read Silver Delta table

df_gold = spark.table(
    "railway_data_engineering.silver.railway_cleaned"
)

print("Silver records available:", df_gold.count())

display(df_gold.limit(10))

In [0]:
# ============================================
# GOLD TABLE 1 — STATION SUMMARY
# ============================================

station_summary = df_gold.groupBy(
    "Source_Station_Name"
).agg(
    count("*").alias("Total_Trains"),
    round(
        avg("day_number"),
        2
    ).alias("Avg_Day_Number")
).orderBy(
    col("Total_Trains").desc()
)

display(station_summary.limit(10))

In [0]:
station_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "railway_data_engineering.gold.station_summary"
    )

print("✅ Gold station_summary created!")

In [0]:
# ============================================
# GOLD TABLE 2 — ROUTE SUMMARY
# ============================================

route_summary = df_gold.groupBy(
    "Source_Station_Name",
    "Destination_Station_Name"
).agg(
    count("*").alias("Total_Services")
).orderBy(
    col("Total_Services").desc()
)

display(route_summary.limit(10))

In [0]:
route_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "railway_data_engineering.gold.route_summary"
    )

print("✅ Gold route_summary created!")

In [0]:
# ============================================
# GOLD TABLE 3 — DAY SUMMARY
# ============================================

day_summary = df_gold.groupBy(
    "days_clean",
    "day_number"
).agg(
    count("*").alias("Train_Count")
).orderBy(
    "day_number"
)

display(day_summary)

In [0]:
day_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "railway_data_engineering.gold.day_summary"
    )

print("✅ Gold day_summary created!")

In [0]:
# ============================================
# GOLD TABLE 4 — DAY TYPE SUMMARY
# ============================================

day_type_summary = df_gold.groupBy(
    "day_type"
).agg(
    count("*").alias("Train_Count")
).orderBy(
    col("Train_Count").desc()
)

display(day_type_summary)

In [0]:
day_type_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "railway_data_engineering.gold.day_type_summary"
    )

print("✅ Gold day_type_summary created!")

In [0]:
# ============================================
# VERIFY GOLD TABLES
# ============================================

gold_tables = [
    "railway_data_engineering.gold.station_summary",
    "railway_data_engineering.gold.route_summary",
    "railway_data_engineering.gold.day_summary",
    "railway_data_engineering.gold.day_type_summary"
]

for table_name in gold_tables:
    df_check = spark.table(table_name)
    print(
        f"{table_name} → {df_check.count()} records"
    )